# STREME / TOMTOM / FIMO pipeline

This notebook configures, runs and summarizes the motif pipeline. The reusable
implementation is in `streme_pipeline.py`; the same code is used by the Slurm
array jobs.


## Setup


In [4]:
from pathlib import Path
import pandas as pd

from streme_pipeline import (
    PipelineConfig,
    PipelineParameters,
    STAGES,
    build_status,
    combine_result_tables,
    discover_fastas,
    export_fimo_for_binding_bench,
    run_all_stages,
    run_pipeline_for_fasta,
    save_analysis_tables,
    save_summaries,
)


In [5]:
config = PipelineConfig.default(
    project=Path("/s/project/ml4rg_students/2026/project15")
)
config.validate()

fastas = discover_fastas(config.fasta_dir)
parameters = PipelineParameters(
    streme_time=1800,
    minw=6,
    maxw=20,
    nmotifs=10,
    fimo_thresh="1e-4",
    fimo_max_stored_scores=100_000,
    fimo_skip_matched_sequence=False,
)

print(f"Found {len(fastas)} FASTA files")
print("FASTA directory:", config.fasta_dir)
print("Result directory:", config.result_dir)
print("JASPAR database:", config.jaspar_fungi)


Found 1401 FASTA files
FASTA directory: /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas
Result directory: /s/project/ml4rg_students/2026/project15/working/streme_results
JASPAR database: /s/project/ml4rg_students/2026/project15/working/jaspar/JASPAR2026_CORE_fungi_non-redundant_pfms_meme.txt


The pipeline writes a `.pipeline_done.json` next to each successful
result. New results are only reused when their command and input signatures
match. Existing results from the old notebook have no manifest and are accepted
as a legacy cache by default.


## Status


In [23]:
!squeue -j 19470729

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)


In [24]:
!sacct -X -j 19470729 --format=JobID,JobName,State,ExitCode,Elapsed,MaxRSS

JobID           JobName      State ExitCode    Elapsed     MaxRSS 
------------ ---------- ---------- -------- ---------- ---------- 
19470729_0   streme-st+  COMPLETED      0:0   00:00:03            
19470729_1   streme-st+  COMPLETED      0:0   00:00:03            
19470729_2   streme-st+  COMPLETED      0:0   00:00:03            
19470729_3   streme-st+  COMPLETED      0:0   00:00:03            
19470729_4   streme-st+  COMPLETED      0:0   00:00:03            
19470729_5   streme-st+  COMPLETED      0:0   00:00:03            
19470729_6   streme-st+  COMPLETED      0:0   00:00:03            
19470729_7   streme-st+  COMPLETED      0:0   00:00:03            
19470729_8   streme-st+  COMPLETED      0:0   00:00:03            
19470729_9   streme-st+  COMPLETED      0:0   00:00:02            
19470729_10  streme-st+  COMPLETED      0:0   00:00:02            
19470729_11  streme-st+  COMPLETED      0:0   00:00:02            
19470729_12  streme-st+  COMPLETED      0:0   00:00:03        

In [19]:
status = build_status(config, fastas)
display(status.head())
display(status[list(STAGES)].sum().rename("completed"))

incomplete = status.loc[~status[list(STAGES)].all(axis=1)]
print(f"Incomplete datasets: {len(incomplete)}")
display(incomplete.head(20))


,name,fasta,streme,tomtom,fimo_jaspar,fimo_streme
0,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
1,_candida_auris_gca_001189475_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
2,_candida_auris_gca_002775015_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
3,_candida_auris_gca_003013715_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
4,_candida_auris_gca_003014415_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True


streme         487
fimo_jaspar     74
tomtom          62
fimo_streme     24
Name: completed, dtype: int64

Incomplete datasets: 1377


,name,fasta,streme,tomtom,fimo_jaspar,fimo_streme
23,absidia_repens_gca_002105175_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
24,acaromyces_ingoldii_gca_003144295_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
25,acidomyces_richmondensis_bfw_gca_001592465_seq...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
26,acidomyces_sp_richmondensis_gca_001572075_sequ...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
27,acremonium_chrysogenum_atcc_11550_gca_00076926...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
28,agaricus_bisporus_var_burnettii_jb137_s8_gca_0...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
29,agrocybe_aegerita_gca_902728275_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
31,akanthomyces_lecanii_rcef_1005_gca_001636795_s...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
32,allomyces_macrogynus_atcc_38327_gca_000151295_...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
33,alternaria_alternata_gca_001642055_sequence_ma...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False


## Test one FASTA


In [4]:
test_fasta = fastas[0]
! /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper.fasta
test_result = run_pipeline_for_fasta(
    config,
    test_fasta,
    parameters=parameters,
    force=False,
    accept_legacy=True,
)
test_result


/bin/bash: /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper.fasta: Permission denied
[_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper] streme: cached
[_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper] fimo_jaspar: existing legacy result accepted
[_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper] tomtom: existing legacy result accepted
[_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper] fimo_streme: existing legacy result accepted


{'streme': 'cached',
 'fimo_jaspar': 'legacy',
 'tomtom': 'legacy',
 'fimo_streme': 'legacy'}

## Run locally or in an interactive allocation

Stages are processed separately. This keeps independent FIMO-JASPAR work from
waiting behind STREME in the same worker. With `max_workers=None`, the pipeline
uses `SLURM_CPUS_PER_TASK` when available and otherwise at most four workers.
Set `max_workers` explicitly when the allocation or available memory requires it.


In [4]:
run_report = run_all_stages(
    config,
    fastas,
    parameters=parameters,
    force=False,
    accept_legacy=False,
    max_workers=2,
)

display(run_report.groupby(["stage", "status"]).size())
failed_report = run_report.loc[run_report["status"] == "failed"]
with pd.option_context("display.max_colwidth", None):
    display(failed_report.head(20))


Stage streme: 1401 FASTAs, max_workers=2
[_candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper] streme: cached
streme: 1/1401 _candida_arabinofermentans_nrrl_yb_2248_gca_001661425_sequence_mapper.fasta -> cached
[_candida_auris_gca_002775015_sequence_mapper] streme: cached
streme: 2/1401 _candida_auris_gca_002775015_sequence_mapper.fasta -> cached
[_candida_auris_gca_001189475_sequence_mapper] streme: cached
streme: 3/1401 _candida_auris_gca_001189475_sequence_mapper.fasta -> cached
[_candida_auris_gca_003013715_sequence_mapper] streme: cached
streme: 4/1401 _candida_auris_gca_003013715_sequence_mapper.fasta -> cached
[_candida_auris_gca_003014415_sequence_mapper] streme: cached
streme: 5/1401 _candida_auris_gca_003014415_sequence_mapper.fasta -> cached
[_candida_auris_gca_008275145_sequence_mapper] streme: cached
streme: 6/1401 _candida_auris_gca_008275145_sequence_mapper.fasta -> cached
[_candida_auris_gca_007168705_sequence_mapper] streme: cached
streme: 7/1401 _can

KeyboardInterrupt: 

import importlib
import streme_pipeline

importlib.reload(streme_pipeline)

fasta = next(
    f for f in fastas
    if f.name == "_candida_auris_gca_008275145_sequence_mapper.fasta"
)

result = streme_pipeline.run_stage_for_fasta(
    config,
    fasta,
    "streme",
    parameters=parameters,
    force=True,
    accept_legacy=False,
)

print(result)

## Recommended: submit Slurm arrays

Run from the repository root in an environment that provides Python and pandas:

```bash
MAX_CONCURRENT=20 bash slurm/submit_streme_arrays.sh
```

The submission script determines the FASTA count automatically. STREME and
FIMO-JASPAR start independently; TOMTOM and FIMO-STREME depend on the
corresponding STREME array task. Set `PYTHON_BIN` before submission if `python`
does not point to the project environment.


## Refresh status


In [20]:
status = build_status(config, fastas)
display(status[list(STAGES)].sum().rename("completed"))
display(status.loc[~status[list(STAGES)].all(axis=1)].head(20))


streme         166
fimo_jaspar     74
tomtom          62
fimo_streme     24
Name: completed, dtype: int64

,name,fasta,streme,tomtom,fimo_jaspar,fimo_streme
23,absidia_repens_gca_002105175_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
24,acaromyces_ingoldii_gca_003144295_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
25,acidomyces_richmondensis_bfw_gca_001592465_seq...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
26,acidomyces_sp_richmondensis_gca_001572075_sequ...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
27,acremonium_chrysogenum_atcc_11550_gca_00076926...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
28,agaricus_bisporus_var_burnettii_jb137_s8_gca_0...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
29,agrocybe_aegerita_gca_902728275_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
31,akanthomyces_lecanii_rcef_1005_gca_001636795_s...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
32,allomyces_macrogynus_atcc_38327_gca_000151295_...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
33,alternaria_alternata_gca_001642055_sequence_ma...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False


## Save summaries

By default, FIMO is reduced to hit counts per dataset and motif. This avoids
loading or rewriting all FIMO hits, which can be very large. TOMTOM results are
small enough to combine into one table.


In [22]:
summary_paths = save_summaries(
    config,
    fastas,
    combine_fimo=False,
)
summary_paths


{'status': PosixPath('/s/project/ml4rg_students/2026/project15/working/streme_results/summary_tables/meme_pipeline_status.csv'),
 'tomtom': PosixPath('/s/project/ml4rg_students/2026/project15/working/streme_results/summary_tables/tomtom_all.tsv'),
 'fimo_jaspar_counts': PosixPath('/s/project/ml4rg_students/2026/project15/working/streme_results/summary_tables/fimo_jaspar_counts.tsv'),
 'fimo_streme_counts': PosixPath('/s/project/ml4rg_students/2026/project15/working/streme_results/summary_tables/fimo_streme_counts.tsv')}

In [23]:
tomtom_path = summary_paths["tomtom"]
if tomtom_path:
    tomtom_preview = pd.read_csv(tomtom_path, sep="\t", nrows=30)
    display(tomtom_preview.sort_values(["dataset", "q-value"]).head(30))

for key in ("fimo_jaspar_counts", "fimo_streme_counts"):
    path = summary_paths[key]
    if path:
        print(key)
        display(pd.read_csv(path, sep="\t", nrows=20))


,dataset,Query_ID,Target_ID,Optimal_offset,p-value,E-value,q-value,Overlap,Query_consensus,Target_consensus,Orientation
28,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,5-RATKACATAATCAA,MA0284.3,-1,0.000001,0.000266,0.000510,10,GATGACATAATCAA,ATTACATAAT,+
29,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,5-RATKACATAATCAA,MA0418.2,-2,0.000009,0.001810,0.001733,10,GATGACATAATCAA,TTACGTAATC,-
0,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,1-ATATATATATATATAT,MA0386.2,1,0.000048,0.009264,0.009166,10,ATATATATATATATAT,GATATATATAT,+
1,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,1-ATATATATATATATAT,MA0379.1,-6,0.000530,0.102347,0.042716,5,ATATATATATATATAT,ATATA,+
2,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,1-ATATATATATATATAT,MA0369.2,0,0.000671,0.129521,0.042716,14,ATATATATATATATAT,TTCTATAAATAGAT,+
5,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,3-CGCGTCGATCG,MA1437.2,-4,0.000397,0.076593,0.152328,7,CGCGTCGATCG,TCGATCG,-
6,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,3-CGCGTCGATCG,MA2699.1,0,0.000875,0.168851,0.167905,11,CGCGTCGATCG,CTCGTCGATCCT,-
14,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,4-ACATCTTTRCAC,MA0359.3,0,0.000627,0.121093,0.240726,12,ACATCTTTACAC,ACACCCATACAT,-
7,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,3-CGCGTCGATCG,MA0329.2,0,0.005023,0.969379,0.530376,5,CGCGTCGATCG,CGCGT,-
8,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,3-CGCGTCGATCG,MA0434.2,-3,0.005600,1.080800,0.530376,7,CGCGTCGATCG,GTAGATC,+


fimo_jaspar_counts


,dataset,motif_id,hit_count
0,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,MA0386.2,13290
1,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,MA0378.2,6391
2,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,MA0390.2,5682
3,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,MA0284.3,4852
4,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,MA2706.1,4717
5,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,MA0398.2,4701
6,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,MA0277.1,4502
7,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,MA2695.1,4311
8,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,MA0388.1,3740
9,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,MA0296.2,3417


fimo_streme_counts


,dataset,motif_id,hit_count
0,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,1-ATATATATATATATAT,25122
1,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,7-AATAAATAAATAAAA,20853
2,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,6-RAAAAAAAAAAAAA,17529
3,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,2-CATCATCATCATCA,14723
4,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,16-ABTATATATATAVT,13638
5,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,17-TAATAATAATAATAA,10135
6,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,12-AAAWTTGAAAAATT,9580
7,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,18-TCATCTTCAAC,7913
8,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,20-AACAACAACA,7688
9,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,8-ACWACAACAACWAC,7451


### Optional full FIMO tables

Only create these when downstream analysis needs every hit. The function reads
one input file at a time, so memory use stays bounded, but the output can still
be very large.


In [ ]:
# fimo_jaspar_all = combine_result_tables(
#     config,
#     fastas,
#     "fimo_jaspar_tsv",
#     "fimo_jaspar_all.tsv",
# )
# fimo_streme_all = combine_result_tables(
#     config,
#     fastas,
#     "fimo_streme_tsv",
#     "fimo_streme_all.tsv",
# )


## Evaluate results

This section creates analysis-ready tables. FIMO hit counts are normalized by
the total number of scanned bases (`hits_per_mbp`) and by the fraction of
sequences with at least one hit (`sequence_fraction`). TOMTOM matches are
filtered at `q-value <= 0.05`; only the best JASPAR match per STREME motif is
used for the recurrence summaries.

A FIMO hit is evidence for a motif occurrence, not by itself evidence for
biological enrichment or transcription-factor activity. Also note that counts
can be truncated when `--max-stored-scores` is reached.


In [30]:
analysis_paths = save_analysis_tables(
    config,
    fastas,
    tomtom_q_value=0.05,
)
analysis_paths


FASTA statistics: 100/1401
FASTA statistics: 200/1401
FASTA statistics: 300/1401


KeyboardInterrupt: 

In [31]:
overview = pd.read_csv(analysis_paths["dataset_overview"], sep="\t")

display(overview.head())
display(
    overview.groupby("species_name", dropna=False)
    .agg(
        datasets=("dataset", "size"),
        median_sequences=("sequence_count", "median"),
        median_jaspar_hits_per_mbp=("jaspar_fimo_hits_per_mbp", "median"),
        median_matched_streme_motifs=("matched_streme_motifs", "median"),
    )
    .sort_values("datasets", ascending=False)
    .head(30)
)


NameError: name 'analysis_paths' is not defined

### Recurrent known motifs

The first table ranks JASPAR motifs by the number of datasets containing at
least one FIMO hit. Use `median_hits_per_mbp` for comparisons because raw hit
counts are strongly affected by dataset size.


In [ ]:
import matplotlib.pyplot as plt

jaspar_recurrence_path = analysis_paths["fimo_jaspar_recurrence"]
if jaspar_recurrence_path:
    jaspar_recurrence = pd.read_csv(jaspar_recurrence_path, sep="\t")
    display(jaspar_recurrence.head(30))

    top = jaspar_recurrence.head(20).sort_values("datasets_with_hits").copy()
    if "motif_alt_id" in top:
        top["motif_label"] = top["motif_alt_id"].replace("", pd.NA).fillna(top["motif_id"])
    else:
        top["motif_label"] = top["motif_id"]
    ax = top.plot.barh(
        x="motif_label",
        y="datasets_with_hits",
        legend=False,
        figsize=(8, 7),
    )
    ax.set_xlabel("Datasets with at least one FIMO hit")
    ax.set_ylabel("JASPAR motif")
    plt.tight_layout()


### Best TOMTOM annotations

These are known JASPAR motifs repeatedly matched by independently discovered
STREME motifs. The annotated FIMO-STREME recurrence table additionally requires
that the matched de-novo motif occurs in the corresponding sequences.


In [ ]:
for key in ("tomtom_recurrence", "fimo_streme_annotated_recurrence"):
    path = analysis_paths[key]
    if path:
        print(key)
        display(pd.read_csv(path, sep="\t").head(30))


## BindingBench export

BindingBench evaluates genomic binding-position recovery, not motif frequency.
It therefore complements the summary tables above rather than replacing them.
The configured validation benchmark currently provides experimental reference
sites for *Saccharomyces cerevisiae*, so only the matching FASTA should be used.

The exporter converts each FIMO-STREME hit to a one-base genomic position,
uses the STREME motif as `feature_idx`, and uses `-log10(p-value)` as the
descending confidence score expected by BindingBench.


In [ ]:
s_cerevisiae_fastas = [
    fasta
    for fasta in fastas
    if fasta.stem.strip("_").startswith("saccharomyces_cerevisiae_sequence_mapper")
]
if len(s_cerevisiae_fastas) != 1:
    raise ValueError(f"Expected one S. cerevisiae FASTA, found {s_cerevisiae_fastas}")

binding_bench_predictions = export_fimo_for_binding_bench(
    config,
    s_cerevisiae_fastas[0],
    result_key="fimo_streme_tsv",
)
binding_bench_predictions
